# FastAPI による MLflow チャンピオンモデルのサービング

## 前提条件

このノートブックを進める前に、以下が完了していることを確認してください。

- `notebooks/05_champion_challenger/` のノートブックが完了していること
- `IrisClassifier` モデルが MLflow Model Registry に登録されていること
- `@champion` エイリアスがモデルの特定バージョンに設定されていること

> **注意**: このノートブックには実行可能なコードセルがありません。  
> FastAPI サーバーは別プロセスで起動する必要があるため、すべてのコマンドは  
> ターミナルで直接実行してください。

## 学習目標

このノートブックでは以下を学習します。

1. MLflow の `@champion` エイリアスを利用した FastAPI アプリの構成
2. `lifespan` コンテキストマネージャーによる起動時モデルロード
3. `/health` エンドポイントと `/predict` エンドポイントの動作
4. MLflow Server と FastAPI App の 2 サーバー構成の起動方法
5. チャンピオンモデル更新後のアプリ再起動による反映手順

## 1. アーキテクチャ概要

このチュートリアルでは、ローカル環境に 2 つのサーバーを起動して MLflow モデルをサービングします。

```
┌─────────────────────────────────────────────────────────────┐
│                      ローカル環境                             │
│                                                             │
│  ┌──────────────────────────┐      ┌──────────────────────┐ │
│  │   MLflow Server :5000    │      │  FastAPI App :8000   │ │
│  │                          │      │                      │ │
│  │  ・Tracking Server       │ <────│  MLFLOW_TRACKING_URI │ │
│  │  ・Model Registry        │      │  MODEL_NAME          │ │
│  │  ・Artifact Store        │      │                      │ │
│  │    (./mlartifacts)       │      │  GET  /health        │ │
│  │                          │      │  POST /predict       │ │
│  └──────────────────────────┘      └──────────┬───────────┘ │
│                                               │             │
└───────────────────────────────────────────────┼─────────────┘
                                                │
                                           [Client]
                                    curl / Python requests
```

### 構成の特徴

| 項目 | 内容 |
|------|------|
| ストレージ | ローカルファイルシステムのみ（`./mlartifacts`）|
| 外部接続 | 不要（完全ローカル動作）|
| クラウドストレージ | 不使用（S3・Azure Blob・GCS 等は不要）|
| サーバー数 | 2 プロセス（MLflow + FastAPI）|

MLflow Server はモデルの保存・取得・バージョン管理を担当し、FastAPI App は起動時に  
`@champion` エイリアスのモデルをロードしてリクエストを受け付けます。

## 2. `app/main.py` の構成解説

`app/main.py` はプロジェクトルートの `app/` ディレクトリに配置されています。  
以下にコードの主要部分を示します（実行はしません）。

### 2.1 Pydantic データモデル

```python
from pydantic import BaseModel

class PredictRequest(BaseModel):
    features: list[float]          # 入力特徴量のリスト

class PredictResponse(BaseModel):
    prediction: list[float]        # 予測結果
    model_name: str                # モデル名
    model_version: str             # バージョン番号
    alias: str                     # 使用したエイリアス（"champion"）

class HealthResponse(BaseModel):
    status: str                    # "ok"
    tracking_uri: str              # MLflow Tracking URI
    model_name: str
    model_version: str
```

Pydantic v2 を使用しており、リクエスト/レスポンスのバリデーションを自動的に行います。

### 2.2 lifespan コンテキストマネージャー

```python
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # --- 起動時処理 ---
    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
    model_name = os.environ.get("MODEL_NAME")

    # 環境変数が未設定の場合は明確なエラーを出力して終了
    if not tracking_uri:
        print("ERROR: MLFLOW_TRACKING_URI environment variable is not set.", file=sys.stderr)
        sys.exit(1)

    mlflow.set_tracking_uri(tracking_uri)
    client = MlflowClient()

    # @champion エイリアスのバージョン情報を取得
    mv = client.get_model_version_by_alias(model_name, "champion")

    # モデルをロードして app.state に保持
    app.state.model = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")
    app.state.model_version = mv.version
    app.state.model_name = model_name
    app.state.tracking_uri = tracking_uri

    yield  # ← ここでアプリがリクエストを受け付ける

    # --- シャットダウン時処理（必要に応じてクリーンアップ）---
```

`lifespan` は FastAPI の推奨パターンで、アプリ起動時に一度だけモデルをロードします。  
MLflow Server が未起動・モデルが未登録・エイリアス未設定の場合は `sys.exit(1)` で  
明確なエラーメッセージを出力してプロセスを終了します。

### 2.3 `/predict` エンドポイント

```python
@app.post("/predict", response_model=PredictResponse)
async def predict(req: PredictRequest) -> PredictResponse:
    """Run inference using the loaded @champion model."""
    model = app.state.model
    features = np.array(req.features).reshape(1, -1)

    # asyncio.to_thread で同期処理をイベントループをブロックせずに実行
    prediction = await asyncio.to_thread(model.predict, features)

    return PredictResponse(
        prediction=prediction.tolist(),
        model_name=app.state.model_name,
        model_version=app.state.model_version,
        alias="champion",
    )
```

`mlflow.pyfunc` の `model.predict()` は同期関数です。  
`asyncio.to_thread()` を使うことで、推論処理中も他のリクエストを受け付けられます。

## 3. サーバー起動手順

2 つのターミナルを開いて、以下の順番でサーバーを起動します。

### ターミナル 1: MLflow Server の起動

プロジェクトルートディレクトリで実行します。

```bash
conda activate mlflow-tutorial

mlflow server \
  --host 0.0.0.0 \
  --port 5000 \
  --default-artifact-root ./mlartifacts
```

起動に成功すると以下のようなログが表示されます。

```
[INFO] Starting gunicorn 21.x.x
[INFO] Listening at: http://0.0.0.0:5000
```

ブラウザで `http://localhost:5000` を開き、MLflow UI が表示されることを確認してください。

### ターミナル 2: FastAPI App の起動

プロジェクトルートディレクトリで実行します。

```bash
conda activate mlflow-tutorial

MLFLOW_TRACKING_URI=http://localhost:5000 \
MODEL_NAME=IrisClassifier \
uvicorn app.main:app --port 8000
```

起動に成功すると以下のようなログが表示されます。

```
Loaded model 'IrisClassifier' version 2 (@champion)
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
```

> **環境変数について**:  
> `MLFLOW_TRACKING_URI` は FastAPI App がどの MLflow Server に接続するかを指定します。  
> `MODEL_NAME` はロードするモデルの登録名です。  
> これらをコマンドのプレフィックスとして設定することで、そのプロセスにのみ適用されます。

## 4. ヘルスチェック

FastAPI App が正常に起動したら、`/health` エンドポイントで状態を確認します。

### curl を使った確認

```bash
curl http://localhost:8000/health
```

正常な場合のレスポンス例:

```json
{
  "status": "ok",
  "tracking_uri": "http://localhost:5000",
  "model_name": "IrisClassifier",
  "model_version": "2"
}
```

### Swagger UI を使った確認

ブラウザで `http://localhost:8000/docs` を開くと、FastAPI が自動生成した  
インタラクティブな API ドキュメントが表示されます。

## 5. 推論リクエストの送信

### 5.1 curl を使った推論

アイリスデータセットのサンプル（がく片長さ・がく片幅・花びら長さ・花びら幅）で推論します。

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```

レスポンス例（クラス 0 = Setosa と予測した場合）:

```json
{
  "prediction": [0.0],
  "model_name": "IrisClassifier",
  "model_version": "2",
  "alias": "champion"
}
```

### 5.2 Python `requests` を使った推論

Python スクリプトやノートブックから呼び出す場合は以下のようにします。

```python
import requests

url = "http://localhost:8000/predict"
payload = {"features": [5.1, 3.5, 1.4, 0.2]}

response = requests.post(url, json=payload)
response.raise_for_status()

result = response.json()
print(f"予測クラス: {result['prediction']}")
print(f"モデル: {result['model_name']} v{result['model_version']} (@{result['alias']})")
```

出力例:

```
予測クラス: [0.0]
モデル: IrisClassifier v2 (@champion)
```

### 5.3 複数サンプルの一括推論

現在の実装は 1 サンプルずつ処理します。複数サンプルを処理するには、  
ループでリクエストを送信するか、バッチ対応のエンドポイントに拡張してください。

```python
samples = [
    [5.1, 3.5, 1.4, 0.2],  # Setosa
    [6.3, 3.3, 4.7, 1.6],  # Versicolor
    [6.3, 2.8, 5.1, 1.5],  # Virginica
]

for features in samples:
    response = requests.post(url, json={"features": features})
    result = response.json()
    print(f"入力: {features} → 予測: {result['prediction']}")
```

## 6. チャンピオンモデルの更新と反映手順

新しいモデルを学習・評価して `@champion` エイリアスを更新した後、  
FastAPI App を再起動することで新しいチャンピオンモデルが反映されます。

### 手順

**ステップ 1: 新しいモデルバージョンを Model Registry に登録する**

```python
# notebooks/05_champion_challenger/ で実施
from mlflow import MlflowClient

client = MlflowClient()

# 新バージョンに @champion エイリアスを付与
client.set_registered_model_alias(
    name="IrisClassifier",
    alias="champion",
    version="3",  # 新しいバージョン番号
)
print("@champion エイリアスをバージョン 3 に更新しました")
```

**ステップ 2: FastAPI App を再起動する**

ターミナル 2 で `Ctrl+C` で停止してから再起動します。

```bash
# Ctrl+C で停止した後
MLFLOW_TRACKING_URI=http://localhost:5000 \
MODEL_NAME=IrisClassifier \
uvicorn app.main:app --port 8000
```

再起動ログで新バージョンがロードされたことを確認します。

```
Loaded model 'IrisClassifier' version 3 (@champion)
```

**ステップ 3: ヘルスチェックで確認する**

```bash
curl http://localhost:8000/health
```

```json
{
  "status": "ok",
  "tracking_uri": "http://localhost:5000",
  "model_name": "IrisClassifier",
  "model_version": "3"
}
```

> **ロールバックについて**:  
> 新バージョンに問題があった場合は、`@champion` エイリアスを旧バージョンに  
> 戻してから FastAPI App を再起動するだけでロールバックできます。  
> エイリアスを使った管理の利点は、モデルのバージョン番号を変更するだけで  
> アプリの設定ファイルを変更せずにデプロイできる点です。

## 7. エラー発生時のトラブルシューティング

### FastAPI App 起動時のエラー

| エラーメッセージ | 原因 | 対処法 |
|-----------------|------|--------|
| `MLFLOW_TRACKING_URI environment variable is not set` | 環境変数未設定 | `MLFLOW_TRACKING_URI=http://localhost:5000` を付けて起動 |
| `MODEL_NAME environment variable is not set` | 環境変数未設定 | `MODEL_NAME=IrisClassifier` を付けて起動 |
| `Failed to get @champion alias` | MLflow Server 未起動 または エイリアス未設定 | MLflow Server を起動し、`@champion` エイリアスを設定 |
| `Failed to load model` | アーティファクトが見つからない | `./mlartifacts` ディレクトリがプロジェクトルートに存在するか確認 |

### 接続確認コマンド

```bash
# MLflow Server の動作確認
curl http://localhost:5000/health

# @champion エイリアスの確認
python -c "
import mlflow
from mlflow import MlflowClient
mlflow.set_tracking_uri('http://localhost:5000')
client = MlflowClient()
mv = client.get_model_version_by_alias('IrisClassifier', 'champion')
print(f'@champion → version {mv.version}')
"
```

## まとめ

このノートブックでは、MLflow Model Registry と FastAPI を組み合わせた  
ローカル 2 サーバー構成のサービングパイプラインを学習しました。

### 習得したこと

| 項目 | 内容 |
|------|------|
| アーキテクチャ | MLflow Server + FastAPI の 2 プロセス構成 |
| モデルロード | `lifespan` による起動時の `@champion` モデル自動ロード |
| エンドポイント | `GET /health`（状態確認）・`POST /predict`（推論）|
| 非同期処理 | `asyncio.to_thread()` による同期推論のノンブロッキング実行 |
| モデル更新 | エイリアス更新 → アプリ再起動の簡単な反映手順 |
| ローカル完結 | 外部インターネット・クラウドストレージ不要 |

### チュートリアル全体のまとめ

これで MLflow チュートリアルのすべてのセクションが完了しました。

1. **MLflow コア概念** — Tracking・Models・Model Registry・Projects
2. **scikit-learn 連携** — autolog・手動ロギング・Model Registry 登録
3. **PyTorch 連携** — エポックごとのメトリクスロギング・モデル保存
4. **Optuna HPO** — ハイパーパラメータ最適化と MLflow の統合
5. **チャンピオン/チャレンジャー管理** — エイリアス API による運用管理
6. **FastAPI サービング** — 本番運用を想定した推論 API の構築

MLflow を活用した機械学習の実験管理からモデルサービングまでの  
一連のワークフローを体験できました。実際のプロジェクトでは、  
このローカル構成をベースに必要に応じてクラウド環境へ移行することができます。